# Figuring out BCPNN

In [7]:
import sys
sys.path.insert(0,'..')

In [8]:
from vigipy import *
import pandas as pd

Read the data and only take what we need for the contingency table. Only get first few lines bc data is large

In [2]:
# Source - https://stackoverflow.com/a/69888274
# Posted by David Kaftan
# Retrieved 2026-05-25, License - CC BY-SA 4.0

from pyarrow.parquet import ParquetFile
import pyarrow as pa 

pf = ParquetFile(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet") 
first_n_rows = next(pf.iter_batches(batch_size = 1000)) 
ae_df = pa.Table.from_batches([first_n_rows]).to_pandas() 


In [3]:
ae_df.tail()

,safetyreportid,receivedate,receive_year,receive_quarter,serious,outcome_death,outcome_lifethreat,outcome_hosp,outcome_disab,reporter_country,reporter_qual,age_years,age_stratum,sex,drug_name,drug_name_source,drug_characterization,drug_indication,reaction_pt
995,7419886-1,2011-04-18,2011,2011Q2,1,1,0,0,0,UNITED STATES,5,65.0,geriatric,male,ATROPINE,medicinalproduct,2,PRODUCT USED FOR UNKNOWN INDICATION,MULTI-ORGAN FAILURE
996,7419886-1,2011-04-18,2011,2011Q2,1,1,0,0,0,UNITED STATES,5,65.0,geriatric,male,ATROPINE,medicinalproduct,2,PRODUCT USED FOR UNKNOWN INDICATION,EMOTIONAL DISTRESS
997,7419886-1,2011-04-18,2011,2011Q2,1,1,0,0,0,UNITED STATES,5,65.0,geriatric,male,ATROPINE,medicinalproduct,2,PRODUCT USED FOR UNKNOWN INDICATION,STRESS
998,7419886-1,2011-04-18,2011,2011Q2,1,1,0,0,0,UNITED STATES,5,65.0,geriatric,male,ATROPINE,medicinalproduct,2,PRODUCT USED FOR UNKNOWN INDICATION,ANXIETY
999,7419886-1,2011-04-18,2011,2011Q2,1,1,0,0,0,UNITED STATES,5,65.0,geriatric,male,ATROPINE,medicinalproduct,2,PRODUCT USED FOR UNKNOWN INDICATION,COAGULOPATHY


In [4]:
# ae_df=pd.read_parquet(r"C:\Users\Admin\drug-safety-signal-detection\data\faers_flat.parquet")
ae_df.head()
#['AE', 'name', 'count'] ('date' is optional for longitudinal models)

drug_adverse = ae_df.groupby(["reaction_pt","drug_name"]).size().reset_index(name="count")

drug_adverse["AE"] = drug_adverse["reaction_pt"].astype(str)
drug_adverse["name"] = drug_adverse["drug_name"].astype(str)
drug_adverse["count"] = drug_adverse["count"].astype(int)

drug_adverse = drug_adverse[["AE", "name", "count"]]


In [5]:
drug_adverse.head()

,AE,name,count
0,ACUTE MYOCARDIAL INFARCTION,ATROPINE,1
1,ACUTE MYOCARDIAL INFARCTION,MORPHINE,1
2,ANHEDONIA,ACETYLSALICYLIC ACID SRT,1
3,ANHEDONIA,ALBUMIN (HUMAN),1
4,ANHEDONIA,ALTACE,1


Un sacco di problemi con la funzione che crea la contingency table usata in convert, per cui sovrascrivo con una che funzia x me ora:

In [ ]:
import numpy as np
import vigipy.utils.data_prep as data_prep

def fixed_compute_contingency(data_frame, product_label, count_label, ae_label, margin_threshold):
    data_cont = pd.pivot_table(
        data_frame,
        values=count_label,
        index=product_label,
        columns=ae_label,
        aggfunc="sum",
        fill_value=0,
    )
    data_cont = data_cont.astype(float)
    data_cont.index = pd.Index(data_cont.index.astype(str).tolist())
    data_cont.columns = pd.Index(data_cont.columns.astype(str).tolist())

    # Usa boolean mask invece di np.where per evitare il problema PyArrow
    row_mask = np.sum(data_cont.values, axis=1) < margin_threshold
    col_mask = np.sum(data_cont.values, axis=0) < margin_threshold
    
    drop_rows = data_cont.index[row_mask]
    drop_cols = data_cont.columns[col_mask]
    data_cont = data_cont.drop(drop_rows)
    data_cont = data_cont.drop(drop_cols, axis=1)
    return data_cont

data_prep.compute_contingency = fixed_compute_contingency

Creo il dataset x benino che vuole lui per convert

In [ ]:
drug_adverse = (
    ae_df.groupby(["drug_name", "reaction_pt"])
    .size()
    .reset_index(name="count")
)

drug_adverse = pd.DataFrame({
    "name":  np.array(drug_adverse["drug_name"], dtype=str),
    "AE":    np.array(drug_adverse["reaction_pt"], dtype=str),
    "count": np.array(drug_adverse["count"], dtype=np.int64),
})

data = convert(drug_adverse)

Funziona! caccio dentro bcpnn, the article on Iapatinib sets IC>0, lower limit of CI>0, N>=3

In [ ]:
res=bcpnn(container=data, min_events=3, decision_metric="rank", ranking_statistic="quantile")
# figure out cosa devo fare per fare quello che hanno fatto per iapatinib

AttributeError: 'Container' object has no attribute 'head'